# Sesgos en los datos

En esta sesión vamos a **revisar** situaciones donde los datos pueden llevarnos a conclusiones incorrectas. Veremos cómo detectar estos problemas antes de entrenar un modelo.

### Contenido de la sesión
1.  **Sesgo Algorítmico**: Cómo los modelos reproducen patrones históricos de los datos.
2.  **Paradoja de Simpson**: Cuando la tendencia global contradice las tendencias por subgrupos.
3.  **Sesgo de Supervivencia**: Analizar sólo lo que "sobrevive" puede llevar a conclusiones erróneas.
4.  **Sesgo de Selección**: Cuando la muestra no representa adecuadamente a la población.
5.  **Data Leakage**: Usar variables que contienen información del futuro.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Configuración visual
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Sesgo Algorítmico y Reproducción de Patrones Históricos

Los modelos de Machine Learning aprenden de los datos históricos, incluyendo los sesgos que estos contengan. 

**El problema:** Si usáis datos históricos que reflejan desigualdades estructurales, el modelo reproducirá estos patrones, incluso sin usar explícitamente variables sensibles.

Veamos un ejemplo de disparidad en la concesión de créditos:
*   Tenemos dos grupos demográficos: A (Privilegiado) y B (Desfavorecido).
*   El modelo no usa la variable "Grupo", pero sí usa el "Score Crediticio".
*   El Score Crediticio, por razones estructurales, tiende a ser más bajo en el grupo B.


In [ ]:
# Generación de datos sintéticos simulando Sesgo Histórico
np.random.seed(123)
n = 10000

# Grupo A (Mayoritario) vs Grupo B (Minoritario)
# 75% Grupo A, 25% Grupo B
grupo = np.random.choice(["A", "B"], n, p=[0.75, 0.25])

# Variables: Score Crediticio (0-1000)
# Simulamos una disparidad estructural: el Grupo B tiene una media menor
# debido a factores sistémicos, aunque hay individuos excelentes en ambos grupos.


prob_pago = np.where(grupo == "A", np.random.beta(12, 5, n), np.random.beta(10, 6, n))
score = prob_pago * 550 + 300


# El Target real (¿Devuelve el préstamo?) depende del SCORE, no del GRUPO explícitamente.
# Un buen pagador es un buen pagador, sea A o B.
paga = np.random.binomial(1, prob_pago)

df_fairness = pd.DataFrame({"Grupo": grupo, "Score": score, "Paga": paga})

# Visualizamos la distribución
plt.figure(figsize=(8, 5))
sns.histplot(data=df_fairness, x="Score", hue="Grupo", kde=True, element="step")
plt.title("Distribución de Score Crediticio por Grupo")
plt.show()

# Entrenamos un modelo "ciego" (No usamos la columna Grupo)
# Solo usamos 'Score' para predecir 'Paga'
X = df_fairness[["Score"]]
y = df_fairness["Paga"]

model_fair = LogisticRegression()
model_fair.fit(X, y)

print(
    "Queremos una tasa de impago inferior al 15%, por lo que usamos un umbral de 0.85"
)

# Predecimos para todos
df_fairness["Probabilidad"] = model_fair.predict_proba(X)[:, 1]
df_fairness["Aprobación"] = df_fairness["Probabilidad"] > 0.85

print("¿Qué proporción de impagos prevemos?")
proporcion_impagos = (
    1 - df_fairness[df_fairness["Aprobación"] == 1]["Probabilidad"].mean()
)
print(proporcion_impagos)

In [ ]:
# --- AUDITORÍA DE FAIRNESS ---
print("--- Tasa de Aprobación (Acceptance Rate) ---")
resumen = df_fairness.groupby("Grupo")["Aprobación"].mean()
print(resumen)

tasa_A = resumen["A"]
tasa_B = resumen["B"]

print(f"\nDisparidad (Ratio B/A): {tasa_B / tasa_A:.2f} (1.0 ideal)")
print(
    "El modelo es matemáticamente correcto (usa sólo el Score), pero el Score ya contiene el sesgo histórico."
)

In [ ]:
# Una posible mitigación: Calibración por Grupos (Threshold Adjustment)
# Se puede ajustar el umbral de decisión según el grupo para compensar el sesgo
df_fairness["Aprobación"] = 0


# Umbral de 0.8 para B (compensación del sesgo histórico)
# pero entonces tenemos que aumentar el umbral para A para mantener la misma proporción de impagos
df_fairness.loc[
    (df_fairness.Grupo == "A") & (df_fairness.Probabilidad > 0.86), "Aprobación"
] = 1
df_fairness.loc[
    (df_fairness.Grupo == "B") & (df_fairness.Probabilidad > 0.82), "Aprobación"
] = 1

tasa_B_adj = df_fairness[df_fairness.Grupo == "B"].Aprobación.mean()
print(
    f"\nDisparidad tras Ajuste de Umbral: {tasa_B_adj / tasa_A:.2f} (más equilibrado)"
)

print("¿Qué proporción de impagos prevemos?")
proporcion_impagos = (
    1 - df_fairness[df_fairness["Aprobación"] == 1]["Probabilidad"].mean()
)
print(proporcion_impagos)
print("")

## 2. La Paradoja de Simpson

Esta paradoja ocurre cuando una tendencia aparece en varios grupos de datos pero **desaparece o se invierte** cuando estos grupos se combinan.

### El Caso Clásico: Admisiones en UC Berkeley (1973)
La universidad fue demandada por discriminación de género. Las cifras globales mostraban que la tasa de admisión de hombres era mucho mayor que la de mujeres.
¿Había discriminación sistemática? Vamos a recrear los datos para verlo.

In [ ]:
# RE-CREACIÓN DATASET UC BERKELEY (Simplificado a 2 Departamentos)
# Departamento A: Muy fácil entrar (Tasa ~80%). Ofertado mayoritariamente por HOMBRES.
# Departamento F: Muy difícil entrar (Tasa ~7%). Ofertado mayoritariamente por MUJERES.

data_berkeley = []

# --- DEPARTAMENTO A (Fácil) ---
# Hombres aplican masivamente aquí
# 800 Hombres aplican, 600 entran (75%)
data_berkeley.extend([{"Genero": "Hombre", "Depto": "A", "Admitido": 1}] * 600)
data_berkeley.extend([{"Genero": "Hombre", "Depto": "A", "Admitido": 0}] * 200)

# Mujeres aplican poco aquí
# 100 Mujeres aplican, 80 entran (80%)
data_berkeley.extend([{"Genero": "Mujer", "Depto": "A", "Admitido": 1}] * 80)
data_berkeley.extend([{"Genero": "Mujer", "Depto": "A", "Admitido": 0}] * 20)


# --- DEPARTAMENTO F (Difícil) ---
# Hombres aplican poco aquí
# 100 Hombres aplican, 6 entran (6%)
data_berkeley.extend([{"Genero": "Hombre", "Depto": "F", "Admitido": 1}] * 6)
data_berkeley.extend([{"Genero": "Hombre", "Depto": "F", "Admitido": 0}] * 94)

# Mujeres aplican masivamente aquí
# 600 Mujeres aplican, 48 entran (8%)
data_berkeley.extend([{"Genero": "Mujer", "Depto": "F", "Admitido": 1}] * 48)
data_berkeley.extend([{"Genero": "Mujer", "Depto": "F", "Admitido": 0}] * 552)


df_simpson = pd.DataFrame(data_berkeley)

print("Datos de Berkeley generados.")
df_simpson.sample(5)

### Análisis Agregado
Si miramos los datos totales sin tener en cuenta el departamento, las cifras parecen claras.
¿Qué porcentaje de Hombres vs Mujeres son admitidos en total?

In [ ]:
print("--- TASA DE ADMISIÓN GLOBAL ---")
tasa_global = df_simpson.groupby("Genero")["Admitido"].mean()
print(tasa_global)

# Visualización engañosa
plt.figure(figsize=(6, 4))
sns.barplot(x="Genero", y="Admitido", data=df_simpson, errorbar=None, palette="viridis")
plt.title(
    f"Visión Global: Hombres {tasa_global['Hombre']:.1%} vs Mujeres {tasa_global['Mujer']:.1%}"
)
plt.ylabel("Tasa de Admisión")
plt.show()

print("Los hombres tienen el doble de tasa de admisión que las mujeres.")

### Análisis Desagregado
La universidad argumentó que las diferencias se debían a que hombres y mujeres solicitaban plazas en departamentos distintos.
Vamos a desglosar la admisión **por Departamento**.

In [ ]:
print("--- TASA DE ADMISIÓN POR DEPARTAMENTO ---")
print(df_simpson.groupby(["Depto", "Genero"])["Admitido"].mean())

# Gráfico revelador
plt.figure(figsize=(8, 5))
sns.barplot(
    x="Depto",
    y="Admitido",
    hue="Genero",
    data=df_simpson,
    errorbar=None,
    palette="viridis",
)
plt.title(
    "Paradoja de Simpson en Berkeley: Las mujeres tienen MAYOR tasa en cada depto"
)
plt.ylabel("Tasa de Admisión")
plt.show()

**Explicación:**
1.  En el Depto A (Fácil), las mujeres tienen mejor tasa de admisión (80% vs 75%).
2.  En el Depto F (Difícil), las mujeres tienen mejor tasa de admisión (8% vs 6%).
3.  **¿Entonces por qué el global es peor?** Porque la mayoría de las mujeres solicitaron el Depto F (el más difícil), lo que arrastra su promedio general hacia abajo. 

**Conclusión:** Las métricas agregadas pueden ser engañosas cuando hay subgrupos con características muy distintas. Hay que controlar por la variable de confusión (en este caso, la dificultad del departamento).

## 3. Sesgo de Supervivencia (Survivorship Bias)

![Survivorship Bias](https://upload.wikimedia.org/wikipedia/commons/b/b2/Survivorship-bias.svg)

> **Historia:** Los aviones de la Segunda Guerra Mundial volvían con muchos agujeros de bala en las alas y cola, pero ninguno en el motor.
> 
> **Intuición inicial:** "Reforcemos las alas y la cola, que es donde les dan".
> 
> **Abraham Wald (Estadístico):** "¡No! Reforzad el motor. Si un avión recibe un tiro en el motor, **no vuelve**. Solo estamos viendo los datos de los supervivientes".

**Ejercicio Mental:**
Si analizamos datos de "Empresas Tecnológicas Más Exitosas en 2025" para buscar patrones de éxito:
1. ¿Qué empresas faltan?
2. Si concluimos que "Tomar riesgos extremos lleva al éxito", estaríamos obviando que muchas empresas que quebraron también tomaron riesgos extremos, pero no están en el dataset.

In [ ]:
# Simulamos un dataset de Churn
np.random.seed(99)
n = 1000

df_churn = pd.DataFrame(
    {
        "antiguedad_meses": np.random.randint(1, 60, n),
        "factura_mensual": np.random.normal(50, 15, n),
        "edad": np.random.randint(18, 80, n),
        "churn": np.random.choice([0, 1], n, p=[0.8, 0.2]),  # 20% Churn real
    }
)

# --- INTRODUCIENDO EL LEAKAGE ---
# Creamos una variable llamada 'coste_gestion_baja'
# Esta variable lógicamente solo existe (> 0) si el cliente SE HA DADO DE BAJA.
# En la vida real, NO tendríamos este dato antes de que el cliente se vaya.
df_churn["coste_gestion_baja"] = (
    df_churn["churn"] * np.random.normal(100, 10, n) + np.random.normal(0, 5, n) * 0.01
)
df_churn.loc[df_churn["churn"] == 0, "coste_gestion_baja"] = 0

# Split de datos
X = df_churn.drop("churn", axis=1)
y = df_churn["churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

## 4. Sesgo de selección

Cuando la muestra no represena adecuadamente la población porque solo vemos una parte de ella, y la variable que queremos predecir está relacionada con el motivo de observar/no observar cada dato.

### Ejemplo Práctico: Sesgo de Selección en Encuestas

Imaginad que queremos medir la **satisfacción de los clientes** de una app de delivery. Enviamos una encuesta por email después de cada pedido.

**El problema:** Solo responden:
- Los muy satisfechos (quieren elogiar).
- Los muy insatisfechos (quieren quejarse).

Los clientes "neutros" ignoran la encuesta. Si entrenamos un modelo con estas respuestas, ¿será representativo de la población real?

In [ ]:
# Simulamos la POBLACIÓN REAL de clientes (1000 pedidos)
np.random.seed(456)
n_total = 10000

# Satisfacción real distribuida normalmente (1-5 estrellas)
satisfaccion_real = np.random.normal(2.5, 0.75, n_total)
satisfaccion_real = np.clip(satisfaccion_real, 1, 5)

# Probabilidad de responder la encuesta según satisfacción
# Extremos (muy mal o muy bien) responden más
prob_respuesta = np.where(
    satisfaccion_real < 2,  # Muy insatisfechos
    0.8,  # 80% responden
    np.where(
        satisfaccion_real > 4,  # Muy satisfechos
        0.2,  # 20% responden
        0.05,  # Neutros: solo 5% responden
    ),
)

# Generamos quién responde
responde = np.random.binomial(1, prob_respuesta)

# Dataset observado (solo los que respondieron)
df_encuesta = pd.DataFrame({"Satisfaccion": satisfaccion_real, "Respondio": responde})

# Comparación
print("--- POBLACIÓN REAL vs MUESTRA OBSERVADA ---")
print(f"Media Real (todos los clientes): {df_encuesta['Satisfaccion'].mean():.2f}")
print(
    f"Media Observada (solo encuestas): {df_encuesta[df_encuesta.Respondio==1]['Satisfaccion'].mean():.2f}"
)

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Población real
axes[0].hist(
    df_encuesta["Satisfaccion"],
    bins=20,
    edgecolor="black",
    alpha=0.7,
    color="steelblue",
)
axes[0].axvline(
    df_encuesta["Satisfaccion"].mean(),
    color="red",
    linestyle="--",
    linewidth=2,
    label="Media Real",
)
axes[0].set_title("Población Real (Todos los clientes)")
axes[0].set_xlabel("Satisfacción (1-5)")
axes[0].set_ylabel("Frecuencia")
axes[0].legend()

# Muestra sesgada (solo respondieron)
axes[1].hist(
    df_encuesta[df_encuesta.Respondio == 1]["Satisfaccion"],
    bins=20,
    edgecolor="black",
    alpha=0.7,
    color="orange",
)
axes[1].axvline(
    df_encuesta[df_encuesta.Respondio == 1]["Satisfaccion"].mean(),
    color="red",
    linestyle="--",
    linewidth=2,
    label="Media Observada",
)
axes[1].set_title("Muestra Observada (Solo encuestas respondidas)")
axes[1].set_xlabel("Satisfacción (1-5)")
axes[1].set_ylabel("Frecuencia")
axes[1].legend()

plt.tight_layout()
plt.show()

print(
    f"\nLa muestra tiene un sesgo de {df_encuesta[df_encuesta.Respondio==1]['Satisfaccion'].mean() - df_encuesta['Satisfaccion'].mean():.2f} puntos"
)

## 5. Data Leakage (Fugas de Información)

Uno de los errores más comunes en Machine Learning. Ocurre cuando usamos variables en el entrenamiento que **no estarán disponibles** o contienen información del futuro en el momento de hacer predicciones reales.

**Ejemplo:** Predecir Churn (abandono de clientes).
Variable problemática: `minutos_hablados_con_reclamaciones`.
Si esta variable sólo se registra *después* de que el cliente llama para darse de baja, estamos usando información del futuro para predecir el pasado.

### Entrenando el modelo con Leakage

In [ ]:
model_bad = LogisticRegression()
model_bad.fit(X_train, y_train)
y_pred_bad = model_bad.predict(X_test)
acc_bad = accuracy_score(y_test, y_pred_bad)

print(f"Accuracy Modelo CON Leakage: {acc_bad:.4f}")
print(
    "Parece increíble, pero el modelo está haciendo trampas sin que nos hayamos dado cuenta."
)

# Miremos qué variable pesa más
coefs = pd.DataFrame({"Feature": X.columns, "Importance": model_bad.coef_[0]})
print("\nImportancia de Variables:")
print(coefs.sort_values(by="Importance", ascending=False))

### Entrenando el modelo Realista (Sin Leakage)
Eliminamos la variable que contiene información del futuro.

In [ ]:
X_train_good = X_train.drop("coste_gestion_baja", axis=1)
X_test_good = X_test.drop("coste_gestion_baja", axis=1)

model_good = LogisticRegression()
model_good.fit(X_train_good, y_train)
y_pred_good = model_good.predict(X_test_good)
acc_good = accuracy_score(y_test, y_pred_good)

print(f"Accuracy Modelo Realista: {acc_good:.4f}")
print(
    "Este resultado está cerca del baseline del 80% (porcentaje de clientes que no hacen churn), pero es realista."
)
print(
    "Ahora sabemos que necesitamos mejores variables, no variables que contengan información del futuro."
)

# Bonus
Cómo funciona una red neuronal

https://playground.tensorflow.org/